In [3]:
import copy

In [4]:
class SecondTick:
    HIGH_AMOUNT = 5000000
    def __init__(self, code):
        self.code = code
        self.reset() 

    def reset(self):
        self.time = None
        self.open = 0
        self.close = 0
        self.high = 0
        self.low = 0
        self.volume = 0
        self.amount = 0
        self.count = 0
        self.buy_high_amount_count = 0
        self.sell_high_amount_count = 0
        self.buy_amount = 0
        self.sell_amount = 0

    def add_volume(self, v):
        self.volume += v

    def set_open(self, p):
        if self.open == 0:
            self.open = p

    def set_close(self, p):
        self.close = p

    def set_high(self, p):
        if p > self.high:
            self.high = p

    def set_low(self, p):
        if self.low == 0 or p < self.low:
            self.low = p

    def add_amount(self, a):
        self.amount += a
        self.count += 1

    def is_empty(self):
        return self.open == 0

    def finish(self, t, current_buy_high_amount, current_sell_high_amount, buy_amount, sell_amount):
        self.time = t
        self.buy_high_amount_count = current_buy_high_amount
        self.sell_high_amount_count = current_sell_high_amount
        self.buy_amount = buy_amount
        self.sell_amount = sell_amount
        c = copy.copy(self)
        self.reset()
        return c

In [1]:
class TickTracker:
    def __init__(self, code, yesterday_close, limit_time, callback):
        self.code = code
        self.tick_time = None
        self.search_date = search_date
        self.second_tick = SecondTick(code)
        self.second_ticks = []
        self.yesterday_close = yesterday_close
        self.open = 0
        self.limit_time = limit_time
        self.is_skip = False
        self.callback = callback
        self.high = 0
        self.low = 0
        self.skip_first_tick = True
        self.buy_amount = 0
        self.sell_amount = 0
        self.buy_high_amount_count = 0
        self.sell_high_amount_count = 0
        self.trend_time = None

    def push_second_tick(self, tick_time, ask_price):
        if self.second_tick.is_empty():
            return

        self.second_ticks.append(self.second_tick.finish(tick_time,
                                                         self.buy_high_amount_count,
                                                         self.sell_high_amount_count,
                                                         self.buy_amount,
                                                         self.sell_amount))
        self.tick_time = tick_time

        if len(self.second_ticks) >= 100 and ask_price > 0:   
            if self.trend_time is None or tick_time - self.trend_time > timedelta(seconds=10):
                self.callback(self.second_ticks, tick_time, ask_price)
                self.trend_time = tick_time

    def set_prices(self, price):
        if self.low == 0 or self.low < price:
            self.low = price

        if price > self.high:
            self.high = price

    def handle_tick(self, t):
        if t['time'] > self.limit_time:
            return
        
        if self.tick_time is None:
            self.tick_time = t['date']

        if t['market_type'] == 50:
            if self.open == 0:
                self.open = t['current_price']
                
            if self.skip_first_tick:
                self.skip_first_tick = False
                return
                
            self.set_prices(t['current_price'])

            if t['date'] - self.tick_time >= timedelta(seconds=20):
                self.push_second_tick(t['date'], t['ask_price'])
            self.second_tick.set_open(t['bid_price'])
            self.second_tick.set_close(t['bid_price'])
            self.second_tick.add_volume(t['volume'])
            amount = t['volume'] * t['current_price']
            self.second_tick.add_amount(amount)
            if t['buy_or_sell'] == 49:
                self.buy_amount += amount
            else:
                self.sell_amount += amount
            self.second_tick.set_high(t['bid_price'])
            self.second_tick.set_low(t['bid_price'])
            
            if amount >= SecondTick.HIGH_AMOUNT:
                if t['buy_or_sell'] == 49:
                    self.buy_high_amount_count += 1
                else:
                    self.sell_high_amount_count += 1
        else:
            self.push_second_tick(t['date'], 0)
            self.skip_first_tick = True
        